# AdaIn Implementation

In [1]:
!pip install -q datasets torchvision tqdm|

/bin/bash: -c: line 2: syntax error: unexpected end of file


In [2]:
import os
import torch
from torchvision import transforms
from PIL import Image
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 1. Define Standard Preprocessing Transformations
# For style transfer, we typically resize and center-crop to a uniform square (e.g., 256x256 or 512x512)
IMAGE_SIZE = 256
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    # VGG networks expect inputs normalized this way if you're using pre-trained weights
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create an infinite generator for real training (No `list()` conversion!)
class AdaINStreamingDataset(torch.utils.data.IterableDataset):
    def __init__(self, hf_stream, transform=None):
        super().__init__()
        self.hf_stream = hf_stream
        self.transform = transform

    def __iter__(self):
        for item in self.hf_stream:
            try:
                img = item.get('image', None)
                if img is None:
                    img = next((val for val in item.values() if isinstance(val, Image.Image)), None)
                if img is None:
                    continue

                img = img.convert('RGB')
                if self.transform:
                    img = self.transform(img)
                yield img
            except Exception:
                continue

# For real training, remove '.take()' and do not wrap in list()
train_content_stream = load_dataset("DavidPhilips/coco2017", split="train", streaming=True)
train_style_stream = load_dataset("huggan/wikiart", split="train", streaming=True)

content_dataset = AdaINStreamingDataset(train_content_stream, transform=transform)
style_dataset = AdaINStreamingDataset(train_style_stream, transform=transform)

# Feed directly to loaders
content_loader = DataLoader(content_dataset, batch_size=64)
style_loader = DataLoader(style_dataset, batch_size=64)

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

dataset_infos.json:   0%|          | 0.00/5.91k [00:00<?, ?B/s]

In [3]:


import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class AdaIN(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def get_stats(self, feat):
        """Calculates channel-wise mean and standard deviation across spatial dimensions."""
        N, C, H, W = feat.size()
        # Flatten spatial dimensions to (N, C, H*W)
        feat_flat = feat.view(N, C, -1)

        mean = feat_flat.mean(dim=2).view(N, C, 1, 1)
        # Adding epsilon prevents division-by-zero during normalization
        std = feat_flat.std(dim=2).view(N, C, 1, 1) + self.eps
        return mean, std

    def forward(self, content_feat, style_feat):
        content_mean, content_std = self.get_stats(content_feat)
        style_mean, style_std = self.get_stats(style_feat)

        # Normalize content and scale/shift with style stats
        normalized = (content_feat - content_mean) / content_std
        return (normalized * style_std) + style_mean

In [4]:
class VGGDecoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Mirrored VGG-19 structure starting from relu4_1 (512 channels) down to RGB (3 channels)
        self.decoder = nn.Sequential(
            # Block 4
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(512, 256, kernel_size=3),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'), # Replaces Pool3

            # Block 3
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, kernel_size=3),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, kernel_size=3),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, kernel_size=3),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 128, kernel_size=3),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'), # Replaces Pool2

            # Block 2
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(128, 128, kernel_size=3),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(128, 64, kernel_size=3),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'), # Replaces Pool1

            # Block 1
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(64, 64, kernel_size=3),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(64, 3, kernel_size=3) # Output normalized RGB pixels
        )

    def forward(self, x):
        return self.decoder(x)

In [5]:
class VGGEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Load pre-trained weights
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features

        # We slice VGG-19 into blocks to extract style features at relu1_1, relu2_1, relu3_1, and relu4_1
        self.slice1 = nn.Sequential(*vgg[:2])   # Up to relu1_1
        self.slice2 = nn.Sequential(*vgg[2:7])  # Up to relu2_1
        self.slice3 = nn.Sequential(*vgg[7:12]) # Up to relu3_1
        self.slice4 = nn.Sequential(*vgg[12:21])# Up to relu4_1 (the bottleneck)

        # Freeze all parameters
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x):
        h1 = self.slice1(x)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        h4 = self.slice4(h3)
        # Returns both intermediate features (for style loss) and the final bottleneck feature
        return [h1, h2, h3, h4]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

# --- Configuration Hyperparameters ---
config= {}

config["epochs"] = 2
config["learning_rate"] = 1e-4
config["style_weight"] = 1.0   # Standard weight from the AdaIN paper (gamma)
config["content_weight"] = 100.0
config["checkpoint_path"]= None # Content loss weight
SAVE_INTERVAL = 100
LOG_INTERVAL = 50
# --- Setup Models & Devices ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config["device"]= DEVICE
# instantiating the individual pieces:
encoder = VGGEncoder().to(config["device"]).eval() # Keep frozen
decoder = VGGDecoder().to(config["device"]).train()
adain = AdaIN().to(config["device"])

# 4. Load the weights using weights mapping
# weights_only=True is a secure practices feature added in newer PyTorch versions
if config["checkpoint_path"] is not None:
    state_dict = torch.load(config["checkpoint_path"], map_location=config["device"], weights_only=True)
    decoder.load_state_dict(state_dict)

optimizer = optim.Adam(decoder.parameters(), lr=config["learning_rate"])

# --- Training Loss Helpers ---
def calc_content_loss(out_features, target_features):
    return F.mse_loss(out_features, target_features)

def calc_style_loss(out_features_list, style_features_list):
    loss = 0.0
    for out_feat, style_feat in zip(out_features_list, style_features_list):
        out_mean, out_std = adain.get_stats(out_feat)
        style_mean, style_std = adain.get_stats(style_feat)

        mean_loss = F.mse_loss(out_mean, style_mean)
        std_loss = F.mse_loss(out_std, style_std)
        loss += mean_loss + std_loss
    return loss

# --- Main Training Loop (Clean Printing) ---
global_step = 0

for epoch in range(config["epochs"]):
    print(f"\n================ Epoch {epoch+1}/{EPOCHS} ================")

    # Zip the streaming loaders together
    train_loader = zip(content_loader, style_loader)

    for content_images, style_images in train_loader:
        # 1. Move batch to GPU
        content_images = content_images.to(config["device"])
        style_images = style_images.to(config["device"])

        optimizer.zero_grad()

        # 2. Extract features
        with torch.no_grad():
            content_feats = encoder(content_images)
            style_feats = encoder(style_images)
            t = adain(content_feats[-1], style_feats[-1])# takes in the last layer coming from the encoder and then returns the stylized features.

        # 3. Decode back to stylized pixels
        stylized_images = decoder(t)
        # 4. Re-encode for losses
        stylized_feats = encoder(stylized_images)

        # 5. Compute losses
        loss_c = calc_content_loss(stylized_feats[-1], t)
        loss_s = calc_style_loss(stylized_feats[:-1], style_feats[:-1])
        total_loss = (config["content_weight"] * loss_c) + (config["style_weight"] * loss_s)
        # 6. Backward and Optimize
        total_loss.backward()
        optimizer.step()

        global_step += 1

        # --- Clean Line-by-Line Console Prints ---
        if global_step % LOG_INTERVAL == 0:
            print(
                f"Step {global_step:6d} | "
                f"Total Loss: {total_loss.item():.4f} | "
                f"Content Loss: {loss_c.item():.4f} | "
                f"Style Loss: {loss_s.item():.4f}"
            )

        # --- Periodic Checkpoint Saving ---
        if global_step % SAVE_INTERVAL == 0:
            checkpoint_path = f"decoder_step_{global_step}.pth"
            torch.save(decoder.state_dict(), checkpoint_path)
            print(f"--> Saved checkpoint: {checkpoint_path}")

# Final save
torch.save(decoder.state_dict(), "decoder_final.pth")
print("\n🎉 Training completed! Final weights saved as 'decoder_final.pth'")


================ Epoch 1/2 ================


In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torch
from torch.utils.data import DataLoader

# --- 1. Grab the very next batch from the streams ---
# We use next(iter(...)) to safely pull one batch from the streaming zip
content_loader = DataLoader(content_dataset, batch_size=32)
style_loader = DataLoader(style_dataset, batch_size=32)
single_batch_loader = zip(content_loader, style_loader)

content_batch, style_batch = next(iter(single_batch_loader))

# --- 2. Select the 3rd image (index 2) and add batch dimension ---
# Models expect shape [1, C, H, W], so we use unsqueeze(0)
c_img = content_batch[30].unsqueeze(0).to(DEVICE)
s_img = style_batch[6].unsqueeze(0).to(DEVICE)

# --- 3. Pass through the trained network ---
encoder.eval()
decoder.eval()  # Make sure decoder is in eval mode

with torch.no_grad():
    # Extract features
    c_feats = encoder(c_img)
    s_feats = encoder(s_img)

    # Stylize features in latent space
    t = adain(c_feats[-1], s_feats[-1])

    # Optional alpha control layer trick (uncomment if you want to bring back content structure):
    alpha = 1
    t = alpha * t + (1 - alpha) * c_feats[-1]

    # Decode features back into stylized pixels
    stylized_tensor = decoder(t)

# --- 4. Convert Tensor back to Image and Visualise ---
# Remove batch dimension [1, C, H, W] -> [C, H, W] and send to CPU
stylized_tensor = stylized_tensor.squeeze(0).cpu()
content_tensor = content_batch[30].cpu()  # Fixed index from 3 to 2 to match c_img
style_tensor = style_batch[6].cpu()      # Fixed index from 3 to 2 to match s_img

# Separate the inverse normalization out so we can clamp the raw tensors
inv_normalize = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

# Apply inverse normalization to the tensors
stylized_tensor = inv_normalize(stylized_tensor)
content_tensor = inv_normalize(content_tensor)
style_tensor = inv_normalize(style_tensor)

# CRITICAL FIX: Clamp values while they are still Tensors to prevent color artifacts
stylized_tensor = torch.clamp(stylized_tensor, 0.0, 1.0)
content_tensor = torch.clamp(content_tensor, 0.0, 1.0)
style_tensor = torch.clamp(style_tensor, 0.0, 1.0)

# Convert safely clamped tensors to viewable PIL images
to_pil = T.ToPILImage()
final_stylized_img = to_pil(stylized_tensor)
original_content_img = to_pil(content_tensor)
original_style_img = to_pil(style_tensor)

# Plot them side-by-side to see your results!
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original_content_img)
axes[0].set_title("Original Content")
axes[0].axis('off')

axes[1].imshow(original_style_img)
axes[1].set_title("Original Style")
axes[1].axis('off')

axes[2].imshow(final_stylized_img)
axes[2].set_title("Final Stylized Output")
axes[2].axis('off')

plt.show()

# Save the image to your disk
final_stylized_img.save("final_stylized_output.png")

In [ ]:
import torch

# 1. Setup the device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Instantiate the exact same decoder architecture structure
decoder = VGGDecoder().to(DEVICE)

# 3. Define the path to the specific checkpoint file you want to reload
# This can be one of your periodic steps or the final file:
checkpoint_path = "decoder_step_3400.pth"  # Or e.g., "decoder_step_5000.pth"

# 4. Load the weights using weights mapping
# weights_only=True is a secure practices feature added in newer PyTorch versions
state_dict = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
decoder.load_state_dict(state_dict)

print(f"✅ Successfully reloaded decoder weights from: {checkpoint_path}")

# 5. Crucial: Put the decoder in evaluation mode if you are running inference!
decoder.eval()

✅ Successfully reloaded decoder weights from: decoder_step_3400.pth


VGGDecoder(
  (decoder): Sequential(
    (0): ReflectionPad2d((1, 1, 1, 1))
    (1): Conv2d(512, 256, kernel_size=(3, 3), stride=(1, 1))
    (2): ReLU()
    (3): Upsample(scale_factor=2.0, mode='nearest')
    (4): ReflectionPad2d((1, 1, 1, 1))
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1))
    (6): ReLU()
    (7): ReflectionPad2d((1, 1, 1, 1))
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1))
    (9): ReLU()
    (10): ReflectionPad2d((1, 1, 1, 1))
    (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1))
    (12): ReLU()
    (13): ReflectionPad2d((1, 1, 1, 1))
    (14): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1))
    (15): ReLU()
    (16): Upsample(scale_factor=2.0, mode='nearest')
    (17): ReflectionPad2d((1, 1, 1, 1))
    (18): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1))
    (19): ReLU()
    (20): ReflectionPad2d((1, 1, 1, 1))
    (21): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1))
    (22): ReLU()
    (23): Upsample(scale_factor=2.0